# Structured output validation with Pydantic and Guardrails AI

We will validate a refund **proposal**, not execute it. The lab deliberately separates structural validation from business authorization.

In [ ]:
from pathlib import Path
from typing import Literal
import importlib.util
import json

import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

In [ ]:
class RefundDecision(BaseModel):
    model_config = ConfigDict(extra="forbid")
    action: Literal["answer", "draft_refund", "request_human"]
    order_id: str = Field(pattern=r"^ord_[a-z0-9]+$")
    amount_inr: int | None = Field(default=None, ge=0, le=100_000)
    explanation: str = Field(min_length=5, max_length=240)

    @model_validator(mode="after")
    def amount_matches_action(self):
        if self.action == "answer" and self.amount_inr is not None:
            raise ValueError("answer must not carry a refund amount")
        if self.action in {"draft_refund", "request_human"} and self.amount_inr is None:
            raise ValueError("refund-related action requires amount_inr")
        return self

candidates = {
    "valid": '{"action":"draft_refund","order_id":"ord_42","amount_inr":450,"explanation":"Within the documented return policy"}',
    "wrong_type": '{"action":"draft_refund","order_id":"ord_42","amount_inr":"nine hundred","explanation":"Refund requested"}',
    "unknown_tool": '{"action":"run_shell","order_id":"ord_42","amount_inr":10,"explanation":"Execute a command"}',
    "smuggled_field": '{"action":"answer","order_id":"ord_42","explanation":"Looks normal","shell":"rm -rf /"}',
    "cross_field": '{"action":"answer","order_id":"ord_42","amount_inr":999,"explanation":"Contradictory fields"}',
    "invalid_json": '{action: draft_refund}',
}

In [ ]:
def validate_candidate(name: str, raw: str) -> dict:
    try:
        parsed = RefundDecision.model_validate_json(raw)
        return {"name": name, "validation_passed": True, "validated": parsed.model_dump(), "error": None}
    except (ValidationError, json.JSONDecodeError) as exc:
        return {"name": name, "validation_passed": False, "validated": None, "error": str(exc)}

results = [validate_candidate(name, raw) for name, raw in candidates.items()]
pd.DataFrame(results)[["name", "validation_passed", "error"]]

## Structural success is not policy success

The valid proposal is typed, but whether it is allowed depends on identity, tenant, order state, amount, and approval rules. Keep that decision explicit.

In [ ]:
valid = RefundDecision.model_validate_json(candidates["valid"])

def policy(decision: RefundDecision, max_draft_inr: int = 500) -> dict:
    if decision.action == "draft_refund" and (decision.amount_inr or 0) > max_draft_inr:
        return {"outcome": "approval_required", "reason": "AMOUNT_THRESHOLD"}
    if decision.action == "request_human":
        return {"outcome": "approval_required", "reason": "MODEL_ESCALATED"}
    return {"outcome": "allow", "reason": "POLICY_OK"}

print("Typed value:", valid)
print("Separate policy:", policy(valid))
assert policy(valid)["outcome"] == "allow"

## Optional: apply the same schema through Guardrails AI

`Guard.for_pydantic` provides a guard outcome and can orchestrate validation/re-ask flows around an LLM. Here we call `parse` on a fixed candidate so the validation boundary remains observable and deterministic.

In [ ]:
if importlib.util.find_spec("guardrails"):
    from guardrails import Guard
    guard = Guard.for_pydantic(output_class=RefundDecision, name="refund-proposal")
    outcome = guard.parse(candidates["valid"])
    print("validation_passed:", outcome.validation_passed)
    print("validated_output:", outcome.validated_output)
    assert outcome.validation_passed
else:
    print("Guardrails AI is not installed. Run: pip install -r requirements.txt")

In [ ]:
# Release checks
by_name = {row["name"]: row for row in results}
assert by_name["valid"]["validation_passed"]
for name in ["wrong_type", "unknown_tool", "smuggled_field", "cross_field", "invalid_json"]:
    assert not by_name[name]["validation_passed"], name

out = Path("_evidence/04_output_validation.json")
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps({
    "schema": RefundDecision.model_json_schema(),
    "results": results,
    "valid_candidate_policy": policy(valid),
}, indent=2, default=str), encoding="utf-8")
print("PASS: invalid outputs fail closed; valid output still faces policy")
print("Wrote", out.resolve())

## Production decision

Use a bounded re-ask only when a formatting correction is safe and cheap. For security-sensitive ambiguity, ask the user or route to a human rather than letting the model repeatedly reinterpret an authorization-relevant request.